# Utforsk features og velg modellgrunnlag

Denne notebooken bruker feature-metadata aktivt for å hindre target leakage. Målet er å finne et robust første feature-sett for å predikere `target_points` i en kommende kamp.

**Viktig:** Vi bruker bare rader merket `SAFE` som modellfeatures. `UNSAFE_LEAKAGE` skal aldri inn i modellen, og `REVIEW` holdes utenfor til innsamlingstidspunktet er dokumentert. Valideringen deles kronologisk på sesong; tilfeldig train/test-splitt vil gi et for optimistisk bilde.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option('display.max_rows', 120)
pd.set_option('display.max_columns', 120)
sns.set_theme(style='whitegrid')

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
FEATURES_PATH = ROOT / 'data/processed/player_fixture_features.csv'
METADATA_PATH = ROOT / 'data/processed/feature_metadata.csv'
assert FEATURES_PATH.exists(), FEATURES_PATH
assert METADATA_PATH.exists(), METADATA_PATH
print('Prosjektrot:', ROOT)

In [ ]:
features = pd.read_csv(FEATURES_PATH, low_memory=False)
metadata = pd.read_csv(METADATA_PATH)
print(f'Features: {features.shape[0]:,} rader × {features.shape[1]:,} kolonner')
print(f'Metadata: {metadata.shape[0]:,} features')
display(features.head(3))
display(metadata.head(10))

## 1. Forstå metadata og lekkasjerisiko

In [ ]:
summary = (metadata.groupby(['safe_for_prediction', 'timing'], dropna=False)
           .size().rename('antall').reset_index()
           .sort_values(['safe_for_prediction', 'antall'], ascending=[True, False]))
display(summary)
pd.crosstab(metadata.feature_group, metadata.safe_for_prediction).plot.barh(figsize=(10, 9), stacked=True)
plt.title('Featuregrupper etter prediksjonssikkerhet')
plt.xlabel('Antall features'); plt.ylabel(''); plt.tight_layout()

In [ ]:
TARGET = 'target_points'
safe_meta = metadata.loc[metadata.safe_for_prediction.eq('SAFE')].copy()
safe_features = [c for c in safe_meta.feature_name if c in features.columns and c != TARGET]
unsafe_present = [c for c in metadata.loc[metadata.safe_for_prediction.eq('UNSAFE_LEAKAGE'), 'feature_name'] if c in features]
review_present = [c for c in metadata.loc[metadata.safe_for_prediction.eq('REVIEW'), 'feature_name'] if c in features]
print(f'SAFE kandidater i data: {len(safe_features)}')
print(f'Blokkerte lekkasjefeatures: {len(unsafe_present)}')
print(f'REVIEW (ikke brukt automatisk): {len(review_present)}')
assert TARGET in features, f'Mangler målvariabelen {TARGET}'
assert not set(safe_features) & set(unsafe_present)
display(safe_meta.feature_group.value_counts().rename('antall_safe').to_frame())

## 2. Datadekning, målvariabel og tidsakse

In [ ]:
display(features[TARGET].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).to_frame())
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(features[TARGET].dropna(), discrete=True, ax=axes[0])
axes[0].set_title('Fordeling av target_points')
if 'season' in features:
    order = sorted(features.season.dropna().astype(str).unique())
    sns.boxplot(data=features, x='season', y=TARGET, order=order, showfliers=False, ax=axes[1])
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].set_title('Target per sesong')
plt.tight_layout()

for col in ['season', 'position']:
    if col in features:
        display(features.groupby(col, dropna=False)[TARGET].agg(['size','count','mean','std']).round(3))

In [ ]:
coverage = pd.DataFrame({
    'dtype': features[safe_features].dtypes.astype(str),
    'missing_pct': features[safe_features].isna().mean().mul(100),
    'n_unique': features[safe_features].nunique(dropna=True),
    'variance': features[safe_features].select_dtypes(include=np.number).var()
}).join(safe_meta.set_index('feature_name')[['feature_group','transformation','notes']])
coverage = coverage.sort_values(['missing_pct','n_unique'], ascending=[False, True])
display(coverage.head(40).round(3))
print('Helt tomme:', coverage.index[coverage.missing_pct.eq(100)].tolist())
print('Konstante:', coverage.index[coverage.n_unique.le(1)].tolist())

## 3. Kandidater, signal og redundans

Korrelasjon under er kun en enkel screening. Den kan overse ikke-lineære sammenhenger og må ikke brukes på hele datasettet for å velge features før sluttesten. Den er likevel nyttig for å oppdage dubletter og svært like 3/5/10-kampers vinduer.

In [ ]:
numeric_safe = [c for c in safe_features if pd.api.types.is_numeric_dtype(features[c])]
# Et fast utvalg gjør rangkorrelasjon praktisk også på store datasett.
screening_data = features[numeric_safe + [TARGET]].sample(
    n=min(10_000, len(features)), random_state=42)
screening_corr = screening_data.corr(method='spearman')
corr_target = (screening_corr[TARGET]
               .drop(TARGET).dropna().sort_values(key=abs, ascending=False))
corr_table = corr_target.rename('spearman_target').to_frame().join(coverage[['feature_group','missing_pct']])
display(corr_table.head(40).round(3))
corr_table.head(25).sort_values('spearman_target').plot.barh(y='spearman_target', figsize=(9, 8), legend=False)
plt.axvline(0, color='black', lw=.8); plt.title('Sterkeste univariate sammenhenger med target')
plt.tight_layout()

In [ ]:
corr = screening_corr.loc[numeric_safe, numeric_safe].abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
high_corr = (upper.stack().rename('abs_spearman').reset_index()
             .rename(columns={'level_0':'feature_1','level_1':'feature_2'})
             .query('abs_spearman >= 0.95').sort_values('abs_spearman', ascending=False))
display(high_corr.head(60).round(3))
print(f'{len(high_corr)} par har |Spearman| >= 0.95')

## 4. Lag et konservativt første feature-sett

In [ ]:
# Fjern features som er ubrukelige uten å se på target. Behold NaN ellers; modellen kan imputere.
excluded_quality = set(coverage.index[(coverage.missing_pct >= 98) | (coverage.n_unique <= 1)])
candidates = [c for c in numeric_safe if c not in excluded_quality]

# Reduser svært korrelerte par deterministisk: minst missing først, deretter kortere navn.
ordered = sorted(candidates, key=lambda c: (coverage.loc[c, 'missing_pct'], len(c), c))
selected, dropped_redundant = [], []
for col in ordered:
    if any(corr.loc[col, kept] >= 0.98 for kept in selected):
        dropped_redundant.append(col)
    else:
        selected.append(col)

print(f'Numeriske SAFE: {len(numeric_safe)}')
print(f'Etter kvalitetsfilter: {len(candidates)}')
print(f'Etter redundansfilter: {len(selected)}')
display(pd.Series(selected, name='baseline_features').to_frame())

## 5. Tidsriktig baseline og feature-gruppe-ablation

Vi trener på alle tidligere sesonger og tester på siste sesong. `DummyRegressor` er minimumskravet. `HistGradientBoostingRegressor` gir en nyttig ikke-lineær tabulær baseline og håndterer manglende verdier. MAE er lett å tolke i FPL-poeng; RMSE straffer store bom mer.

> Featurevalg og hyperparametre bør senere gjøres med walk-forward-validering *innenfor treningsperioden*. Siste sesong bør forbli urørt som sluttest.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

model_data = features.dropna(subset=[TARGET, 'season']).copy()
seasons = sorted(model_data.season.astype(str).unique())
assert len(seasons) >= 2, 'Trenger minst to sesonger for tidsbasert holdout'
test_season = seasons[-1]
train_mask = model_data.season.astype(str) < test_season
test_mask = model_data.season.astype(str).eq(test_season)
print('Trening:', seasons[:-1], int(train_mask.sum()), 'rader')
print('Test:', test_season, int(test_mask.sum()), 'rader')

def evaluate(cols, label):
    X_train, y_train = model_data.loc[train_mask, cols], model_data.loc[train_mask, TARGET]
    X_test, y_test = model_data.loc[test_mask, cols], model_data.loc[test_mask, TARGET]
    model = DummyRegressor(strategy='mean') if not cols else HistGradientBoostingRegressor(
        learning_rate=.06, max_iter=120, max_leaf_nodes=31, l2_regularization=1.0, random_state=42)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        model.fit(X_train if cols else np.zeros((len(y_train), 1)), y_train)
        pred = model.predict(X_test if cols else np.zeros((len(y_test), 1)))
    return {'modell': label, 'n_features': len(cols),
            'MAE': mean_absolute_error(y_test, pred),
            'RMSE': mean_squared_error(y_test, pred) ** .5,
            'mean_prediction': pred.mean()}, model

results = [evaluate([], 'Dummy: treningsgjennomsnitt')[0]]
results.append(evaluate(selected, 'Alle valgte SAFE-features')[0])
largest_groups = safe_meta[safe_meta.feature_name.isin(selected)].feature_group.value_counts().head(8).index
for group, group_cols in safe_meta[safe_meta.feature_group.isin(largest_groups)].groupby('feature_group').feature_name:
    cols = [c for c in selected if c not in set(group_cols)]
    if cols and len(cols) < len(selected):
        results.append(evaluate(cols, f'Uten gruppe: {group}')[0])
results = pd.DataFrame(results).sort_values('MAE')
display(results.round(3))

## 6. Walk-forward-test og automatisk beslutningsrapport

In [ ]:
folds = []
for validation_season in seasons[1:]:
    tr = model_data.season.astype(str) < validation_season
    va = model_data.season.astype(str).eq(validation_season)
    Xtr, ytr = model_data.loc[tr, selected], model_data.loc[tr, TARGET]
    Xva, yva = model_data.loc[va, selected], model_data.loc[va, TARGET]
    mdl = HistGradientBoostingRegressor(learning_rate=.06, max_iter=120, max_leaf_nodes=31,
                                        l2_regularization=1.0, random_state=42).fit(Xtr, ytr)
    pred = mdl.predict(Xva)
    folds.append({'validation_season': validation_season, 'train_rows': len(ytr),
                  'validation_rows': len(yva), 'MAE': mean_absolute_error(yva, pred),
                  'RMSE': mean_squared_error(yva, pred) ** .5})
walk_forward = pd.DataFrame(folds)
display(walk_forward.round(3))
walk_forward.plot(x='validation_season', y=['MAE','RMSE'], marker='o', figsize=(9,4))
plt.title('Stabilitet over tid'); plt.ylabel('Feil i FPL-poeng'); plt.tight_layout()

In [ ]:
best = results.iloc[0]
recommendation = pd.DataFrame({
    'bruk': selected,
}).join(safe_meta.set_index('feature_name')[['feature_group','transformation']], on='bruk')
print('ANBEFALING')
print('1. Start med kun SAFE-features i tabellen under.')
print('2. Hold UNSAFE_LEAKAGE og REVIEW helt ute av første modell.')
print('3. Bruk sesongbasert walk-forward CV; aldri tilfeldig splitting av kamp-rader.')
print('4. Sammenlign alltid mot DummyRegressor og gjerne en enkel regularisert lineær modell.')
print('5. Velg/juster features på eldre folds; bruk siste sesong én gang som sluttest.')
print(f'Beste holdout-variant i denne kjøringen: {best.modell} (MAE={best.MAE:.3f}).')
display(recommendation.groupby('feature_group').size().rename('antall').to_frame())
display(recommendation)

# Kan brukes direkte av neste notebook/script:
MODEL_FEATURES = recommendation.bruk.tolist()
print('MODEL_FEATURES er nå tilgjengelig med', len(MODEL_FEATURES), 'kolonner.')